# LangGraph 学习
> Web: https://github.langchain.ac.cn/langgraph/

**目标**：  
1、理解LangGraph底层架构任务   
2、构建一个简单智能体并进行部署

In [ ]:
%pip install -U langgraph # Version: 1.2.2

Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
Note: you may need to restart the kernel to use updated packages.
Name: langgraph
Version: 1.2.2
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: /home/huangjie/miniconda3/lib/python3.13/site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip show langgraph

Name: langgraph
Version: 1.2.2
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: /home/huangjie/miniconda3/lib/python3.13/site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## 上下文工程


In [4]:
from openai import OpenAI

client = OpenAI(api_key="sk-acf3a88bb6be4b4aa002e36ac972e9d8",base_url="https://api.deepseek.com")
messages = [
    {
        "role": "system",
        "content": "你是一个专业AI助手"
    }
]

while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break

    messages.append({
        "role": "user",
        "content": user_input
    })
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=0.7
    )
    assistant_reply = response.choices[0].message.content
    messages.append({
        "role": "assistant",
        "content": assistant_reply
    })
    print("User:", user_input)
    print("\nAssistant:", assistant_reply)

User: 我叫：BigYellowJie，我喜欢游泳，有什么推荐游泳装备品牌吗？

Assistant: 你好，BigYellowJie！游泳是一项非常棒的运动，选对装备既能提升表现，也能增加乐趣。以下是一些经过市场验证的优质游泳装备品牌推荐，覆盖不同需求和预算：

### 1. **Speedo（速比涛）**  
   - **核心产品**：泳镜、泳衣/泳裤、泳帽  
   - **特点**：行业标杆，科技感强（如Fastskin系列减少阻力），适合竞技和日常训练。  
   - **推荐理由**：泳镜防雾耐用，泳衣贴合度高，初学者和进阶选手都能找到合适款。

### 2. **Arena（阿瑞娜）**  
   - **核心产品**：竞赛泳衣、训练装备  
   - **特点**：专注专业竞技，面料弹性好，泳镜视野清晰。  
   - **推荐理由**：如果你追求速度或高强度训练，Arena的碳纤维泳衣和防雾镜片很值得投资。

### 3. **TYR**  
   - **核心产品**：铁三/公开水域装备、泳镜  
   - **特点**：美国品牌，设计偏硬核，泳镜防紫外线且防漏效果好。  
   - **推荐理由**：适合喜欢户外游泳或铁人三项的爱好者，性价比高。

### 4. **Zoggs**  
   - **核心产品**：泳镜、泳帽  
   - **特点**：防雾技术领先（如Fog-X科技），佩戴舒适，适合敏感眼睛。  
   - **推荐理由**：如果常因泳镜起雾烦恼，Zoggs的防雾寿命比多数品牌长。

### 5. **FINIS**  
   - **核心产品**：训练器材（划水掌、浮板、脚蹼）  
   - **特点**：专注游泳训练辅助工具，设计科学（如Agility浮板可纠正姿势）。  
   - **推荐理由**：想提升技术或进行陆上模拟训练，FINIS的器材很专业。

### 6. **MP Michael Phelps**  
   - **核心产品**：泳镜、泳衣、训练配件  
   - **特点**：菲尔普斯联名，设计时尚，泳镜贴合度高且抗冲击。  
   - **推荐理由**：颜值在线，适合追求运动与潮流结合的泳者。

### 7. **Decathlon（迪卡侬）**  
   - **核心产品**：入门到中端全系列

In [5]:
messages

[{'role': 'system', 'content': '你是一个专业AI助手'},
 {'role': 'user', 'content': '我叫：BigYellowJie，我喜欢游泳，有什么推荐游泳装备品牌吗？'},
 {'role': 'assistant',
  'content': '你好，BigYellowJie！游泳是一项非常棒的运动，选对装备既能提升表现，也能增加乐趣。以下是一些经过市场验证的优质游泳装备品牌推荐，覆盖不同需求和预算：\n\n### 1. **Speedo（速比涛）**  \n   - **核心产品**：泳镜、泳衣/泳裤、泳帽  \n   - **特点**：行业标杆，科技感强（如Fastskin系列减少阻力），适合竞技和日常训练。  \n   - **推荐理由**：泳镜防雾耐用，泳衣贴合度高，初学者和进阶选手都能找到合适款。\n\n### 2. **Arena（阿瑞娜）**  \n   - **核心产品**：竞赛泳衣、训练装备  \n   - **特点**：专注专业竞技，面料弹性好，泳镜视野清晰。  \n   - **推荐理由**：如果你追求速度或高强度训练，Arena的碳纤维泳衣和防雾镜片很值得投资。\n\n### 3. **TYR**  \n   - **核心产品**：铁三/公开水域装备、泳镜  \n   - **特点**：美国品牌，设计偏硬核，泳镜防紫外线且防漏效果好。  \n   - **推荐理由**：适合喜欢户外游泳或铁人三项的爱好者，性价比高。\n\n### 4. **Zoggs**  \n   - **核心产品**：泳镜、泳帽  \n   - **特点**：防雾技术领先（如Fog-X科技），佩戴舒适，适合敏感眼睛。  \n   - **推荐理由**：如果常因泳镜起雾烦恼，Zoggs的防雾寿命比多数品牌长。\n\n### 5. **FINIS**  \n   - **核心产品**：训练器材（划水掌、浮板、脚蹼）  \n   - **特点**：专注游泳训练辅助工具，设计科学（如Agility浮板可纠正姿势）。  \n   - **推荐理由**：想提升技术或进行陆上模拟训练，FINIS的器材很专业。\n\n### 6. **MP Michael Phelps**  \n   - **核心产品**：泳镜、泳衣、训练配件 